# 🚀 ComfyUI Google Colab One-Click Auto Runner (Bản Tối Ưu Tối Đa)

Notebook này tự động cài đặt và vận hành **ComfyUI** trên Google Colab với hiệu năng tối ưu nhất:
- 💾 **Gắn Google Drive Thông Minh**: Tự động liên kết (symlink) lưu toàn bộ ảnh sinh ra vào `/content/drive/MyDrive/ComfyUI_Output` (không lo mất ảnh khi ngắt session).
- 📦 **Cài Đặt ComfyUI & SD 1.5 Model**: Clone ComfyUI core, cài đặt dependencies và tải sẵn model chuẩn nhẹ `v1-5-pruned-emaonly.safetensors` (~4GB).
- 🌐 **Đường Hầm Kết Nối Đa Nguồn (Dual-Tunnel)**: Chạy song song **Cloudflare Tunnel** và **Localtunnel** làm dự phòng (giải quyết 100% vấn đề nhà mạng Viettel/VNPT/FPT chặn link).

👉 **Hướng dẫn sử dụng**: Chọn **Runtime -> Run all** (hoặc bấm tổ hợp phím `Ctrl + F9`) để chạy từ A đến Z!

In [ ]:
# @title 1. Gắn Google Drive & Tối ưu lưu trữ Output
import os

USE_DRIVE = True

if USE_DRIVE:
    try:
        from google.colab import drive
        print("📁 Đang kết nối với Google Drive...")
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive", force_remount=False)
        
        if os.path.exists("/content/drive/MyDrive"):
            drive_output = "/content/drive/MyDrive/ComfyUI_Output"
            os.makedirs(drive_output, exist_ok=True)
            print(f"✅ Đã kết nối Google Drive! Ảnh sẽ tự động lưu vào: {drive_output}")
        else:
            print("⚠️ Chưa gắn được Google Drive (bỏ qua cấp quyền). Sẽ dùng bộ nhớ tạm Colab.")
    except Exception as e:
        print(f"⚠️ Lưu ý: {e}. Tiến hành dùng bộ nhớ tạm của Colab.")
else:
    print("ℹ️ Bỏ qua gắn Google Drive. Sử dụng thư mục local Colab.")


In [ ]:
# @title 2. Cài đặt ComfyUI Core & Tải Model Stable Diffusion v1.5
import os
import subprocess

%cd /content

# 1. Clone ComfyUI repo nếu chưa có
if not os.path.exists("/content/ComfyUI"):
    print("📦 Clone ComfyUI từ GitHub...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI

# 2. Xử lý liên kết (symlink) thư mục output sang Google Drive
drive_output_path = "/content/drive/MyDrive/ComfyUI_Output"
if os.path.exists(drive_output_path):
    if os.path.exists("/content/ComfyUI/output") and not os.path.islink("/content/ComfyUI/output"):
        !rm -rf /content/ComfyUI/output
    if not os.path.exists("/content/ComfyUI/output"):
        !ln -s "/content/drive/MyDrive/ComfyUI_Output" /content/ComfyUI/output
        print("🔗 Đã liên kết (symlink) thư mục output với Google Drive!")
else:
    os.makedirs("/content/ComfyUI/output", exist_ok=True)
    print("📁 Đã tạo thư mục output lưu trữ local tại /content/ComfyUI/output.")

# 3. Cài đặt requirements & công cụ hỗ trợ
print("📥 Cài đặt thư viện dependencies...")
!apt-get update -qq && apt-get install -y -qq aria2 nodejs npm
!pip install -r requirements.txt --quiet
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --quiet

# 4. Tải Model SD 1.5 pruned-emaonly (sử dụng link tối ưu tránh lỗi 403)
model_path = "/content/ComfyUI/models/checkpoints/v1-5-pruned-emaonly.safetensors"
if not os.path.exists(model_path):
    print("⬇️ Đang tải model Stable Diffusion v1.5 pruned-emaonly (~4GB)...")
    !aria2c --console-log-level=error --summary-interval=0 -c -x 4 -s 4 -k 1M "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors?download=true" -d /content/ComfyUI/models/checkpoints -o v1-5-pruned-emaonly.safetensors || \
     wget -c -O {model_path} "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors?download=true"
    print("✅ Đã tải thành công Model SD 1.5!")
else:
    print("✅ Model SD 1.5 đã sẵn sàng.")


In [ ]:
# @title 3. Khởi động ComfyUI & Khởi tạo Đường Hầm Public (Cloudflare & Localtunnel)
import os
import time
import subprocess
import re
import urllib.request

# 1. Tải và cài đặt Cloudflared nếu chưa có
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📦 Đang cài đặt Cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    !rm -f cloudflared-linux-amd64.deb
    print("✅ Đã cài đặt Cloudflared!")

%cd /content/ComfyUI

# 2. Khởi chạy ComfyUI Core ở background
comfy_cmd = "python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header"
comfy_log = "/content/comfyui.log"

if os.path.exists(comfy_log):
    os.remove(comfy_log)

subprocess.Popen(f"{comfy_cmd} > {comfy_log} 2>&1", shell=True)
print("⏳ Đang khởi động ComfyUI Server (vui lòng đợi vài giây cho ComfyUI nạp model).../")

# Polling kiểm tra ComfyUI sẵn sàng tại port 8188
comfy_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/", timeout=2) as response:
            if response.status == 200:
                comfy_ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if comfy_ready:
    print("✅ ComfyUI Server đã hoạt động hoàn toàn tại 127.0.0.1:8188!")
else:
    print("⚠️ Khởi động ComfyUI tốn nhiều thời gian hơn dự kiến, tiến hành tạo đường hầm...")

# 3. Khởi chạy Cloudflare Tunnel ở background
tunnel_cmd = "cloudflared tunnel --url http://127.0.0.1:8188"
tunnel_log = "/content/cloudflared.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)
subprocess.Popen(f"{tunnel_cmd} > {tunnel_log} 2>&1", shell=True)

# 4. Khởi chạy Localtunnel làm đường hầm dự phòng
lt_log = "/content/localtunnel.log"
if os.path.exists(lt_log):
    os.remove(lt_log)
subprocess.Popen(f"npx localtunnel --port 8188 > {lt_log} 2>&1", shell=True)

# Lấy password giải mã của Localtunnel (IP Public Colab)
try:
    colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf-8').strip()
except Exception:
    colab_ip = "N/A"

# 5. Quét log tìm Public URLs
cf_url = None
lt_url = None

for _ in range(25):
    time.sleep(1)
    if not cf_url and os.path.exists(tunnel_log):
        with open(tunnel_log, "r") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                cf_url = m.group(0)
    if not lt_url and os.path.exists(lt_log):
        with open(lt_log, "r") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", f.read())
            if m:
                lt_url = m.group(0)
    if cf_url and lt_url:
        break

print("\n" + "═"*70)
if cf_url:
    print(f"🌐 LINK CHÍNH (Cloudflare Tunnel): {cf_url}")
else:
    print("🌐 LINK CHÍNH (Cloudflare): Đang kết nối, kiểm tra log dưới.")

if lt_url:
    print(f"🔄 LINK DỰ PHÒNG (Localtunnel):     {lt_url}")
    print(f"🔑 Mật khẩu nhập vào Localtunnel:  {colab_ip}")

print("💡 Lưu ý: Nếu mạng Viettel/VNPT/FPT bị chặn trycloudflare.com, hãy dùng Link Dự Phòng!")
print("═"*70 + "\n")

# 6. Live stream log ComfyUI
print("📋 STREAM LOG COMFYUI (Đang hoạt động...):")
try:
    with open(comfy_log, "r") as f:
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng ComfyUI.")
